# 11.7 Statistical Foundation of Time Series

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.07_statistical-foundation-of-time-series.ipynb)

Companion code for Section 11.7 of the book. Run the cells in order from top to bottom.

**Example 11.7.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings; warnings.filterwarnings('ignore')

# ── Monthly air passengers dataset (classic benchmark) ────────────
# 144 months: 1949-1960, strong trend + seasonal
import statsmodels.api as sm
data = sm.datasets.get_rdataset('AirPassengers').data
ts = pd.Series(data['value'].values,
               index=pd.date_range('1949-01', periods=len(data), freq='MS'),
               name='passengers')

print('=== Air Passengers Dataset ===')
print(f'Observations: {len(ts)} monthly from {ts.index.min().date()} to {ts.index.max().date()}')
print(f'Range: {ts.min()} to {ts.max()} passengers')

# Train-test split: last 24 months as test
train = ts[:'1958-12']
test  = ts['1959-01':]
print(f'Train: {len(train)} months, Test: {len(test)} months')

# ── Model 1: SARIMA(1,1,1)(1,1,1)[12] ───────────────────────────
# Fit on log scale (multiplicative seasonality)
log_train = np.log(train)
sarima = SARIMAX(
    log_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima.fit(disp=False)

# Forecast 24 steps
forecast_sarima = np.exp(sarima_fit.forecast(steps=len(test)))
mae_sarima  = mean_absolute_error(test, forecast_sarima)
mape_sarima = np.mean(np.abs((test - forecast_sarima) / test)) * 100
print(f'\nSARIMA(1,1,1)(1,1,1)[12]: MAE={mae_sarima:.1f}  MAPE={mape_sarima:.2f}%')
print(f'AIC: {sarima_fit.aic:.2f}  BIC: {sarima_fit.bic:.2f}')

# ── Ljung-Box test for residual whiteness ─────────────────────────
lb_result = acorr_ljungbox(sarima_fit.resid.dropna(), lags=12, return_df=True)
print(f'Ljung-Box test (residual autocorrelation):')
print(f'  p-values at lags 1,6,12: {lb_result.lb_pvalue.iloc[[0,5,11]].round(4).tolist()}')
print(f'  Good model: all p-values > 0.05 (residuals are white noise)')

# ── Model 2: Holt-Winters Exponential Smoothing ───────────────────
hw = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='mul',   # Multiplicative seasonality (seasonal ∝ level)
    seasonal_periods=12
)
hw_fit = hw.fit(optimized=True)
forecast_hw = hw_fit.forecast(len(test))
mae_hw  = mean_absolute_error(test, forecast_hw)
mape_hw = np.mean(np.abs((test - forecast_hw) / test)) * 100
print(f'\nHolt-Winters (add trend, mul seasonal): MAE={mae_hw:.1f}  MAPE={mape_hw:.2f}%')
print(f'  Alpha (level): {hw_fit.params["smoothing_level"]:.4f}')
print(f'  Beta (trend):  {hw_fit.params["smoothing_trend"]:.4f}')
print(f'  Gamma (season):{hw_fit.params["smoothing_seasonal"]:.4f}')

# ── Naive benchmarks ─────────────────────────────────────────────
# Seasonal naive: forecast = same month last year
naive_seasonal = pd.Series(
    [train.iloc[-(12-i) % 12 - 12] for i in range(len(test))],
    index=test.index
)
mae_naive = mean_absolute_error(test, naive_seasonal)
mape_naive= np.mean(np.abs((test-naive_seasonal)/test))*100
print(f'\nSeasonal Naive (last year same month): MAE={mae_naive:.1f}  MAPE={mape_naive:.2f}%')

# ── Visualise forecasts ───────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Classical Forecasting Models - Air Passengers Dataset',
            fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(train.index, train.values, 'b-', lw=2, label='Training data')
ax.plot(test.index, test.values, 'k-', lw=2, label='Actual test')
ax.plot(test.index, forecast_sarima, 'r--', lw=2.5, label=f'SARIMA MAPE={mape_sarima:.1f}%')
ax.plot(test.index, forecast_hw,     'g--', lw=2.5, label=f'Holt-Winters MAPE={mape_hw:.1f}%')
ax.axvline(train.index[-1], color='gray', ls=':', lw=2, alpha=0.7, label='Forecast origin')

# SARIMA confidence interval
forecast_full = sarima_fit.get_forecast(steps=len(test))
ci = np.exp(forecast_full.conf_int(alpha=0.05))
ax.fill_between(test.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.15, color='red',
               label='SARIMA 95% CI')
ax.set_ylabel('Passengers'); ax.set_title('Forecast Comparison')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Residual plot
ax = axes[1]
resid = sarima_fit.resid
ax.plot(resid.index, resid.values, color='#1976D2', lw=1, alpha=0.8)
ax.axhline(0, color='red', ls='--', lw=1.5)
ax.fill_between(resid.index, -2*resid.std(), 2*resid.std(), alpha=0.1, color='gray',
               label='±2σ band')
ax.set_ylabel('Residual'); ax.set_title('SARIMA Residuals (should be white noise)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('classical_forecasting.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Auto ARIMA order selection ────────────────────────────────────
print('\n=== ARIMA Order Selection via AIC ===')
best_aic = np.inf
best_order = None
for p in range(0, 3):
    for d in range(0, 2):
        for q in range(0, 3):
            try:
                model = ARIMA(log_train, order=(p,d,q))
                res   = model.fit()
                if res.aic < best_aic:
                    best_aic   = res.aic
                    best_order = (p, d, q)
            except: pass
print(f'Best ARIMA order (AIC): {best_order}  AIC={best_aic:.2f}')